In [4]:
#JOB A
def parse_line(line):
    code, dia = line.split(",")
    code = code.strip().upper()
    family = code.split("-")[1]
    return code, family, float(dia)

raw_lines = [
    "MS-FLG-2026-0041, 24.98",
    "AB-XYZ-2025-0007, 10.5",
    "ms-flg-2026-0099 , 18.2"
]

for line in raw_lines:
    code, family, diameter = parse_line(line)
    print(code, family, diameter)


MS-FLG-2026-0041 FLG 24.98
AB-XYZ-2025-0007 XYZ 10.5
MS-FLG-2026-0099 FLG 18.2


In [9]:
def parse_line(line):
    """Take 'MS-FLG-2026-0041, 24.98' → ('MS-FLG-2026-0041', 'FLG', 24.98)"""
    code, dia = line.split(",")
    code = code.strip().upper()
    family = code.split("-")[1]
    return code, family, float(dia)


def build_register(lines):
    """Many raw lines → {family: [readings]}"""
    reg = {}
    for line in lines:
        code, family, dia = parse_line(line)
        if family not in reg:
            reg[family] = []
        reg[family].append(dia)
    return reg


def family_stats(readings):
    """A list of readings → (count, mean, min, max)"""
    if not readings:
        return 0, None, None, None   # decision: no crash, explicit empty result

    n = len(readings)
    mean = sum(readings) / n
    return n, mean, min(readings), max(readings)


# ---- Shift A data ----
shift_A = [
    "MS-FLG-2026-0041, 24.98",
    "MS-FLG-2026-0042, 25.04",
    "MS-FLG-2026-0043, 24.93",
    "AB-XYZ-2025-0007, 10.5",
    "AB-XYZ-2025-0008, 10.7",
    "ms-flg-2026-0099 , 18.2"
]

# ---- Build + Report ----
register = build_register(shift_A)

for family, readings in register.items():
    n, mean, mn, mx = family_stats(readings)
    if mean is None:
        print(f"{family} | n=0 | mean=None | min=None | max=None")
    else:
        print(f"{family} | n={n} | mean={mean:.2f} | min={mn:.2f} | max={mx:.2f}")

FLG | n=4 | mean=23.29 | min=18.20 | max=25.04
XYZ | n=2 | mean=10.60 | min=10.50 | max=10.70


In [10]:
def parse_line(line):
    code, dia = line.split(",")
    code = code.strip().upper()
    family = code.split("-")[1]
    return code, family, float(dia)


def build_register(lines):
    reg = {}
    for line in lines:
        code, family, dia = parse_line(line)
        if family not in reg:
            reg[family] = []
        reg[family].append(dia)
    return reg


def family_stats(readings):
    if not readings:
        return 0, None, None, None
    n = len(readings)
    mean = sum(readings) / n
    return n, mean, min(readings), max(readings)


# ---- Shift A ----
raw_lines = [
    "MS-FLG-2026-0041, 24.98",
    "MS-FLG-2026-0042, 25.04",
    "MS-FLG-2026-0043, 24.93",
    "AB-XYZ-2025-0007, 10.5",
    "AB-XYZ-2025-0008, 10.7",
    "ms-flg-2026-0099 , 18.2"
]

# ---- Shift B ----
shift_B = [
    "MS-FLG-2026-0101, 25.10",
    "MS-FLG-2026-0102, 25.00",
    "AB-XYZ-2025-0010, 10.4",
    "CD-NEW-2026-0001, 30.0"
]


# ---- Build registers ----
regA = build_register(raw_lines)
regB = build_register(shift_B)

# ---- Set operations ----
famA = set(regA.keys())
famB = set(regB.keys())

print("Both shifts :", famA & famB)
print("Only A      :", famA - famB)
print("New in B    :", famB - famA)
print("Whole day   :", famA | famB)


# ---- Drift analysis ----
print("\nDrift Analysis:")

for fam in famA & famB:
    nA, meanA, minA, maxA = family_stats(regA[fam])
    nB, meanB, minB, maxB = family_stats(regB[fam])

    drift = abs(meanB - meanA)
    spreadA = maxA - minA

    print(f"{fam} | meanA={meanA:.2f} | meanB={meanB:.2f} | drift={drift:.2f} | spreadA={spreadA:.2f}")

Both shifts : {'FLG', 'XYZ'}
Only A      : set()
New in B    : {'NEW'}
Whole day   : {'FLG', 'NEW', 'XYZ'}

Drift Analysis:
FLG | meanA=23.29 | meanB=25.05 | drift=1.76 | spreadA=6.84
XYZ | meanA=10.60 | meanB=10.40 | drift=0.20 | spreadA=0.20
